# Anima Heatmap · ComfyUI 실행

GPU 런타임에서 순서대로 실행합니다. 이 노트북은 소스·모델 설치, 서버 준비 확인, 터널 실행만 담당합니다.

기본 설정으로 GitHub의 HisameOgasahara/Anima_HEATMAP 저장소를 clone합니다. 별도 ZIP 업로드 없이 순서대로 실행하세요.

Quick Tunnel 주소를 아는 사람은 ComfyUI에 접속할 수 있습니다. 개인 실험용이며 사용 중에는 실행 셀을 유지하고, 종료할 때 셀의 중지 버튼을 누르세요.


In [ ]:
# 1. 설정
from pathlib import Path
import sys, subprocess, urllib.request, json, shutil, zipfile, tarfile, os

WORK = Path('/content')
COMFY = WORK / 'ComfyUI'
COMFY_REPO = 'https://github.com/comfyanonymous/ComfyUI.git'
COMFY_REF = 'master'  # 재현이 필요하면 사용할 branch 또는 tag를 지정
SOURCE_REPO = 'https://github.com/HisameOgasahara/Anima_HEATMAP.git'  # 비우면 프로젝트 ZIP 업로드
SOURCE_REF = 'feature/tag-influence'
SOURCE_SUBDIR = '.'  # 저장소 루트에 pyproject.toml이 있음
PORT = 8188
MODEL_REPO = 'circlestone-labs/Anima'
MODEL_REVISION = 'main'
MODEL_FILES = {
    'diffusion_models': 'anima-base-v1.0.safetensors',
    'text_encoders': 'qwen_3_06b_base.safetensors',
    'vae': 'qwen_image_vae.safetensors',
}

def run(*args, cwd=None):
    subprocess.run([str(x) for x in args], cwd=cwd, check=True)

def download(url, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file():
        return target
    partial = target.with_suffix(target.suffix + '.partial')
    run('curl', '--fail', '--location', '--retry', '3', '--output', partial, url)
    partial.replace(target)
    return target

server = globals().get('server')
tunnel = globals().get('tunnel')


In [ ]:
# 2. ComfyUI와 분석 모듈 소스 준비
if not COMFY.exists():
    run('git', 'clone', '--depth', '1', '--branch', COMFY_REF, COMFY_REPO, COMFY)
else:
    run('git', 'fetch', '--depth', '1', 'origin', COMFY_REF, cwd=COMFY)
    run('git', 'checkout', '--detach', 'FETCH_HEAD', cwd=COMFY)

if SOURCE_REPO:
    checkout = WORK / 'anima-heatmap-source'
    if not checkout.exists():
        run('git', 'clone', '--depth', '1', '--branch', SOURCE_REF, SOURCE_REPO, checkout)
    else:
        run('git', 'fetch', '--depth', '1', 'origin', SOURCE_REF, cwd=checkout)
        run('git', 'checkout', '--detach', 'FETCH_HEAD', cwd=checkout)
    SOURCE = (checkout / SOURCE_SUBDIR).resolve()
else:
    from google.colab import files
    import uuid
    uploaded = files.upload()
    if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith('.zip'):
        raise ValueError('Anima_Heatmap 프로젝트 ZIP 하나를 업로드하세요.')
    archive = Path(next(iter(uploaded)))
    extracted = WORK / ('anima-source-' + uuid.uuid4().hex)
    extracted.mkdir()
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            target = (extracted / member.filename).resolve()
            if not target.is_relative_to(extracted.resolve()):
                raise ValueError('ZIP에 허용되지 않는 경로가 있습니다.')
        bundle.extractall(extracted)
    candidates = [p.parent for p in extracted.rglob('pyproject.toml') if (p.parent / 'anima_heatmap').is_dir()]
    if len(candidates) != 1:
        raise ValueError('ZIP에서 Anima Heatmap 프로젝트를 하나로 식별할 수 없습니다.')
    SOURCE = candidates[0]
if not (SOURCE / 'anima_heatmap').is_dir():
    raise FileNotFoundError(f'분석 모듈을 찾을 수 없습니다: {SOURCE}')
print('모듈 소스:', SOURCE)


In [ ]:
# 3. 전용 Python 환경과 커스텀 노드 설치
UV_ROOT = WORK / 'anima-uv'
UV_ROOT.mkdir(exist_ok=True)
UV = UV_ROOT / 'uv-x86_64-unknown-linux-gnu' / 'uv'
if not UV.exists():
    archive = download('https://github.com/astral-sh/uv/releases/latest/download/uv-x86_64-unknown-linux-gnu.tar.gz', UV_ROOT / 'uv.tar.gz')
    with tarfile.open(archive) as bundle:
        bundle.extractall(UV_ROOT, filter='data')
ENV = WORK / 'anima-runtime'
PYTHON = ENV / 'bin/python'
if not PYTHON.exists():
    run(UV, 'venv', '--system-site-packages', '--python', sys.executable, ENV)
run(UV, 'pip', 'install', '--python', PYTHON, '-r', COMFY / 'requirements.txt', '-e', SOURCE)
for node_name in ('comfyui_anima_heatmap', 'comfyui_anima_profiler'):
    node_target = COMFY / 'custom_nodes' / node_name
    if node_target.is_symlink():
        if node_target.resolve() != (SOURCE / node_name).resolve():
            node_target.unlink()
            node_target.symlink_to(SOURCE / node_name, target_is_directory=True)
    elif node_target.exists():
        raise FileExistsError(f'기존 노드 폴더를 확인하세요: {node_target}')
    else:
        node_target.symlink_to(SOURCE / node_name, target_is_directory=True)
run(PYTHON, '-c', 'import torch, anima_heatmap; print("CUDA:", torch.cuda.is_available()); print("모듈:", anima_heatmap.__file__); assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."')


In [ ]:
# 4. 모델 다운로드
from urllib.parse import quote
for folder, filename in MODEL_FILES.items():
    print('모델 준비:', filename, flush=True)
    url = f'https://huggingface.co/{MODEL_REPO}/resolve/{quote(MODEL_REVISION, safe="")}/split_files/{folder}/{quote(filename)}'
    download(url, COMFY / 'models' / folder / filename)
print('모델 준비 완료')


In [ ]:
# 태그 영향 지도 예제 워크플로 다운로드
from google.colab import files
workflow = WORK / 'tag_influence.json'
shutil.copyfile(SOURCE / 'example/tag_influence.json', workflow)
files.download(str(workflow))


In [ ]:
# 5. ComfyUI 실행 · Cloudflare 접속 링크 · 로그 유지
import importlib.util
from IPython.display import display, Markdown

spec = importlib.util.spec_from_file_location('anima_notebook_runtime', SOURCE / 'notebooks/runtime.py')
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)

# 이 셀을 다시 실행하면 이전 서버와 터널을 먼저 종료합니다.
for process in (globals().get('tunnel'), globals().get('server')):
    runtime.stop_process(process)
server = tunnel = None
CLOUDFLARED = download('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', WORK / 'cloudflared')
CLOUDFLARED.chmod(0o755)
tunnel_log = WORK / 'anima_tunnel.log'
required = ['AnimaHeatmapTextEncode', 'AnimaHeatmapSettings', 'AnimaHeatmapSampler', 'AnimaHeatmapView', 'AnimaHeatmapLoadSession', 'AnimaTagInfluenceSampler', 'AnimaTagInfluenceView']

try:
    print('ComfyUI 시작 중...', flush=True)
    server, local_url, server_log = runtime.start_server(COMFY, PYTHON, PORT, required_nodes=required)
    print('서버 준비 완료. HTTP/2 터널 연결 중...', flush=True)
    tunnel, public_url = runtime.start_tunnel(CLOUDFLARED, local_url, tunnel_log, server)
    display(Markdown(f'[ComfyUI 열기]({public_url})'))
    print('분석 결과 폴더:', COMFY / 'output/anima_heatmap')
    print('사용 중에는 이 셀을 실행 상태로 두세요. 종료하려면 셀의 중지 버튼을 누르세요.', flush=True)
    runtime.follow_logs(server, tunnel, server_log, tunnel_log)
finally:
    try:
        runtime.stop_process(tunnel)
    finally:
        runtime.stop_process(server)
    server = tunnel = None


## ComfyUI 연결

1. `UNETLoader`에서 Anima, `CLIPLoader`의 stable_diffusion 타입에서 Qwen, `VAELoader`에서 Qwen Image VAE를 선택합니다.
2. 긍정·부정 프롬프트를 각각 `Anima Heatmap · Text Encode`로 인코딩합니다.
3. `Anima Heatmap · Settings`와 모델·조건·latent를 `Anima Heatmap · KSampler`에 연결합니다.
4. sampler의 latent를 VAE Decode로 보낸 뒤, 이미지와 sampler의 session을 `Anima Heatmap · View`에 연결합니다.
5. View의 단어·단계·layer를 바꿔 다시 실행하면 저장한 지도로 다시 분석합니다.

서버/터널 로그는 `/content/ComfyUI/anima_heatmap_server.log`, `/content/anima_tunnel.log`에 있습니다.

## 태그 영향 지도

다운로드한 `tag_influence.json`을 ComfyUI 캔버스에 드래그합니다. 기본 예제는 `red shirt`의 영향을 비교합니다.

1. 원본 긍정 프롬프트를 `positive`에, 비교할 태그만 제거한 긍정 프롬프트를 `ablated_positive`에 연결합니다. 두 프롬프트는 같은 CLIP으로 인코딩합니다.
2. `Tag Influence Sampler`에서 seed, steps, cfg, sampler_name, scheduler, denoise를 설정합니다. 기존 KSampler의 설정을 사용하며 steps는 자유롭게 바꿀 수 있습니다.
3. latent를 VAE Decode에 연결하고, 완성 이미지와 influence를 `Tag Influence View`에 연결합니다.
4. `view=per_step`으로 단계별 지도, `aggregate`로 선택한 단계의 평균을 봅니다. `steps`는 `all`, `last`, `0,4,8`, `0-9`처럼 선택하며 입력 번호는 0부터, 이미지 표시는 1부터 시작합니다.
5. `scale_max=0`이면 전체 측정의 최대값을 공통 색 눈금으로 사용합니다. View 설정만 변경하면 저장된 측정을 다시 표시합니다.

밝은 위치는 같은 생성 상태에서 태그를 제거했을 때 모델의 예측이 크게 달라진 곳입니다. 원본 프롬프트로만 다음 단계에 진행하며 비교의 영향은 누적되지 않습니다. 각 단계의 지도는 최종 완성 이미지에 겹칩니다.

한 단계에서 여러 번 모델을 계산하는 샘플러는 실제 noise schedule 구간으로 측정을 묶습니다. 호출별 지도를 구간 안에서 평균하므로 모델 호출 수와 단계 수가 다를 수 있습니다. 이 기능은 태그를 제거한 프롬프트를 비교합니다. attention 내부 V 차단 실험과는 측정 방식이 다릅니다.

태그 한 개 비교에도 추가 모델 계산이 필요합니다. 같은 모델을 순서대로 사용하고 차이 지도는 CPU 메모리에 보관합니다. 해상도와 스텝 수를 높이면 실행 시간과 메모리 사용량이 증가합니다.
